# IDE Model Configuration — Self-Hosted Model

Configure your IDE to use the **self-hosted model** (qwen36-27b) served via RHOAI MaaS gateway. No external AI API keys required.

> **Prerequisites:**
> - MCP server registration → `../1_mcp_servers/4_connect_ide_clients.ipynb`
> - MaaS API key → `../2_maas/2_enable_maas.ipynb` (MAAS_API_KEY must be set in `.env`)

| IDE | Model Config Method | Status |
|-----|-------------------|--------|
| **Cursor** | Settings → Models → OpenAI API Key + Override Base URL | Cursor Pro+ required |
| **VS Code** | BYOK Custom Endpoint (`chatLanguageModels.json`) | Working |
| **Claude Code** | Environment variables (`ANTHROPIC_BASE_URL`) | ⚠️ Not compatible with RHOAI 3.4 vLLM (see §3) |

> **Self-signed certificate:** Run `launchctl setenv NODE_TLS_REJECT_UNAUTHORIZED 0` once (macOS), then **quit and reopen** your IDE. See `../1_mcp_servers/4_connect_ide_clients.ipynb` for details.

**Sections:**
1. Cursor — Model configuration
2. VS Code — Model configuration (BYOK)
3. Claude Code — Model configuration (env vars)
4. MaaS Gateway — Unified Endpoint
5. Verify connectivity

## 0. Load Environment

In [1]:
import subprocess, json, os
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen36-27b")
MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", f"https://maas-api.{CLUSTER_DOMAIN}/{MODEL_NAMESPACE}/{MODEL_NAME}")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

if MAAS_API_KEY and len(MAAS_API_KEY) > 16:
    api_key_masked = MAAS_API_KEY[:12] + "..." + MAAS_API_KEY[-4:]
else:
    api_key_masked = "<YOUR_MAAS_API_KEY>"

print(f"Cluster:       {CLUSTER_DOMAIN}")
print(f"Model:         {MODEL_NAME}")
print(f"Model URL:     {MODEL_ENDPOINT}/v1")
print(f"MaaS API Key:  {api_key_masked}")

if not MAAS_API_KEY:
    print("")
    print("⚠️  MAAS_API_KEY is not set in .env")
    print("   Run ../2_maas/2_enable_maas.ipynb first to generate an API key.")

Cluster:       apps.openshift-cluster.sandbox1785.opentlc.com
Model:         qwen36-27b
Model URL:     https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b/v1
MaaS API Key:  sk-oai-mZ1qw...YDkz


---
## 1. Cursor IDE — Model Configuration

![Cursor](../images/cursor_integration.png)

In **Cursor Settings** (`Cmd+,`) → **Models** tab:

### Setup Steps

1. Scroll to **API Keys** section
2. **OpenAI API Key** → paste MaaS API key → toggle **ON**
3. **Override OpenAI Base URL** → toggle **ON** → enter `<MODEL_ENDPOINT>/v1`
4. Scroll up → **+ Add Model** → enter `<MODEL_NAME>` → toggle **ON**
5. Select `<MODEL_NAME>` from the Chat model picker

> **Note:** Make sure to open **Cursor Settings**, not VS Code Settings. Click `Cursor Settings` in the left sidebar.
>
> Requires **Cursor Pro or higher**. For team plans, the admin must allow custom model usage. If you see "blocked by team admin settings", contact your team admin.

In [3]:
print("Cursor Model Settings:")
print("=" * 50)
print(f"  Override OpenAI Base URL:  {MODEL_ENDPOINT}/v1")
print(f"  OpenAI API Key:            {api_key_masked}")
print(f"  Model Name:                {MODEL_NAME}")
print("")
print("Steps:")
print("  1. Cmd+, → Cursor Settings (not VS Code Settings)")
print("  2. Models tab → scroll to 'API Keys' section")
print("  3. 'OpenAI API Key' → paste key → toggle ON")
print("  4. 'Override OpenAI Base URL' → toggle ON → paste URL above")
print(f"  5. Scroll up → '+ Add Model' → type '{MODEL_NAME}' → toggle ON")
print(f"  6. Chat model picker → select '{MODEL_NAME}'")

Cursor Model Settings:
  Override OpenAI Base URL:  https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b/v1
  OpenAI API Key:            sk-oai-mZ1qw...YDkz
  Model Name:                qwen36-27b

Steps:
  1. Cmd+, → Cursor Settings (not VS Code Settings)
  2. Models tab → scroll to 'API Keys' section
  3. 'OpenAI API Key' → paste key → toggle ON
  4. 'Override OpenAI Base URL' → toggle ON → paste URL above
  5. Scroll up → '+ Add Model' → type 'qwen36-27b' → toggle ON
  6. Chat model picker → select 'qwen36-27b'


---
## 2. VS Code — Model Configuration (BYOK)

![VS Code](../images/vscode_integration.png)

Use **BYOK (Bring Your Own Key)** to register the self-hosted model. No Copilot subscription or GitHub login required.

### Setup Steps

1. `Cmd+Shift+P` → **`Chat: Manage Language Models`**
2. Select **`Custom Endpoint`** → enter group name (e.g. `RHOAI`)
3. Paste your **MaaS API key** when prompted → VS Code stores it in **Secret Storage**
4. Edit `chatLanguageModels.json` with the model details (see code cell below)
5. To update the key later: **right-click** the RHOAI group → **Update API Key**

> **How API keys work in VS Code BYOK:**
> - The raw API key is stored in VS Code's Secret Storage (encrypted), not in the JSON file
> - The `apiKey` field in JSON shows a reference like `${input:chat.lm.secret.<hash>}` — do NOT replace this with the raw key
> - `requestHeaders` uses `${apiKey}` placeholder, which VS Code replaces with the stored secret at request time
>
> **Token limits**: `maxInputTokens + maxOutputTokens` must not exceed the vLLM `max_model_len` (16384 for qwen36-27b).

In [4]:
vscode_model_config = [
    {
        "name": "RHOAI",
        "vendor": "customendpoint",
        "apiKey": "${input:chat.lm.secret.<hash>}",
        "apiType": "chat-completions",
        "models": [
            {
                "id": MODEL_NAME,
                "name": MODEL_NAME,
                "url": f"{MODEL_ENDPOINT}/v1/chat/completions",
                "toolCalling": True,
                "vision": False,
                "maxInputTokens": 14336,
                "maxOutputTokens": 2048,
                "requestHeaders": {
                    "Authorization": "Bearer ${apiKey}"
                }
            }
        ]
    }
]

print("=== .vscode/chatLanguageModels.json ===")
print(json.dumps(vscode_model_config, indent=2))
print("")
print("⚠️  apiKey field:")
print("   The '${input:chat.lm.secret.<hash>}' value is auto-generated by VS Code")
print("   when you enter the API key via the UI. Do NOT replace it manually.")
print("   To update: right-click RHOAI group → 'Update API Key' → paste key")
print("")
print("📌 requestHeaders:")
print("   'Authorization: Bearer ${apiKey}' ensures the MaaS gateway receives")
print("   the correct auth header. ${apiKey} is replaced at request time.")
print("")
print(f"📌 Token limits: 14336 + 2048 = 16384 (= vLLM max_model_len)")

=== .vscode/chatLanguageModels.json ===
[
  {
    "name": "RHOAI",
    "vendor": "customendpoint",
    "apiKey": "sk-oai-1Bq6X...bk0h",
    "apiType": "chat-completions",
    "models": [
      {
        "id": "qwen36-27b",
        "name": "qwen36-27b (RHOAI)",
        "url": "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b/v1/chat/completions",
        "toolCalling": true,
        "maxInputTokens": 32768,
        "maxOutputTokens": 4096
      }
    ]
  }
]

Steps:
  1. Cmd+Shift+P → 'Chat: Manage Language Models'
  2. Select 'Custom Endpoint' → enter 'RHOAI' → paste API key
  3. Configuration saved to chatLanguageModels.json
  4. Select model from Chat model picker

NOTE: apiKey is masked. Use the full MAAS_API_KEY from .env when configuring.


---
## 3. Claude Code — Model Configuration

![Claude](../images/claude_integration.png)

Claude Code connects to self-hosted models via environment variables. Set these **before** launching `claude` to skip Anthropic login.

| Env Var | Purpose |
|---------|--------|
| `ANTHROPIC_BASE_URL` | MaaS inference endpoint |
| `ANTHROPIC_AUTH_TOKEN` | MaaS API key (**NOT** `ANTHROPIC_API_KEY`) |
| `ANTHROPIC_MODEL` | Model name for inference |
| `MAX_THINKING_TOKENS` | `0` — disable extended thinking |
| `DISABLE_AUTOUPDATER` | `1` — prevent auto-update |

> **Critical:** Use `ANTHROPIC_AUTH_TOKEN`, not `ANTHROPIC_API_KEY`.
>
> **⚠️ Known limitation (RHOAI 3.4 / vLLM 0.18.x):** Claude Code sends `system` role inside the Anthropic Messages API `messages[]` array, but vLLM 0.18.x (shipped with RHOAI 3.4) strictly validates roles as `user` or `assistant` only, returning HTTP 400. This is fixed in vLLM >= 0.23.0 ([vllm-project/vllm#44283](https://github.com/vllm-project/vllm/pull/44283)). **Until RHOAI ships the updated vLLM, Claude Code model integration is not available. Use Cursor or VS Code instead.** MCP tool connections via Claude Code still work normally.

In [4]:
claude_launch_cmd = f"""export ANTHROPIC_BASE_URL="{MODEL_ENDPOINT}"
export ANTHROPIC_AUTH_TOKEN="{api_key_masked}"
export ANTHROPIC_MODEL="{MODEL_NAME}"
export MAX_THINKING_TOKENS="0"
export DISABLE_AUTOUPDATER="1"
claude"""

print("=== Launch Claude Code with self-hosted model ===")
print("")
print(claude_launch_cmd)
print("")
print("Set the environment variables first, then launch claude.")
print("This skips Anthropic login and connects directly to the self-hosted model.")
if not MAAS_API_KEY:
    print("NOTE: Replace the API key placeholder with your actual MAAS_API_KEY.")
else:
    print(f"NOTE: Key is masked. Use full MAAS_API_KEY from .env when copy-pasting.")

=== Launch Claude Code with self-hosted model ===

export ANTHROPIC_BASE_URL="https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b"
export ANTHROPIC_AUTH_TOKEN="sk-oai-mZ1qw...YDkz"
export ANTHROPIC_MODEL="qwen36-27b"
export MAX_THINKING_TOKENS="0"
export DISABLE_AUTOUPDATER="1"
claude

Set the environment variables first, then launch claude.
This skips Anthropic login and connects directly to the self-hosted model.
NOTE: Key is masked. Use full MAAS_API_KEY from .env when copy-pasting.


### Shell Script (Optional — `run-claude.sh`)

In [6]:
script_content = f"""#!/bin/bash
# Launch Claude Code with self-hosted RHOAI model
# Usage: source run-claude.sh

export ANTHROPIC_BASE_URL="{MODEL_ENDPOINT}"
export ANTHROPIC_AUTH_TOKEN="${{MAAS_API_KEY:-{api_key_masked}}}"
export ANTHROPIC_MODEL="{MODEL_NAME}"
export MAX_THINKING_TOKENS="0"
export DISABLE_AUTOUPDATER="1"

echo "Claude Code configured for: ${{ANTHROPIC_MODEL}}"
echo "Endpoint: ${{ANTHROPIC_BASE_URL}}"
claude "$@"
"""

print("=== run-claude.sh ===")
print(script_content)

=== run-claude.sh ===
#!/bin/bash
# Launch Claude Code with self-hosted RHOAI model
# Usage: source run-claude.sh

export ANTHROPIC_BASE_URL="https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b"
export ANTHROPIC_AUTH_TOKEN="${MAAS_API_KEY:-sk-oai-1Bq6X...bk0h}"
export ANTHROPIC_DEFAULT_OPUS_MODEL="qwen36-27b"
export ANTHROPIC_DEFAULT_SONNET_MODEL="qwen36-27b"
export ANTHROPIC_DEFAULT_HAIKU_MODEL="qwen36-27b"
export CLAUDE_CODE_FILE_READ_MAX_OUTPUT_TOKENS="30000"
export CLAUDE_CODE_MAX_OUTPUT_TOKENS="50000"
export MAX_THINKING_TOKENS="0"

echo "Claude Code configured for: ${ANTHROPIC_DEFAULT_SONNET_MODEL}"
echo "Endpoint: ${ANTHROPIC_BASE_URL}"
claude "$@"



---
## 4. MaaS Gateway — Unified Endpoint

The **MaaS Gateway** (configured in Phase 2) provides unified auth for both model and MCP access with a single API key.

| | Direct Route (Phase 1) | MaaS Gateway (Phase 2+) |
|-|--------------------------|-------------------------|
| MCP endpoints | 5 separate URLs | 1 unified URL |
| Authentication | None | API key |
| Rate limiting | None | Per-subscription token limits |
| Config entries | 5 MCP + 1 model | 1 MCP gateway + 1 model |

In [7]:
MAAS_GW = f"https://maas-api.{CLUSTER_DOMAIN}"

print("=== MaaS Gateway Mode ===")
print("")
print("Cursor (.cursor/mcp.json):")
cursor_maas = {
    "mcpServers": {
        "mcp-gateway": {
            "url": f"{MAAS_GW}/mcp/mcp",
            "headers": {"Authorization": f"Bearer {api_key_masked}"}
        }
    }
}
print(json.dumps(cursor_maas, indent=2))

print("")
print("VS Code (.vscode/mcp.json):")
vscode_maas = {
    "servers": {
        "mcp-gateway": {
            "type": "http",
            "url": f"{MAAS_GW}/mcp/mcp",
            "headers": {"Authorization": f"Bearer {api_key_masked}"}
        }
    }
}
print(json.dumps(vscode_maas, indent=2))

print("")
print("Claude Code (CLI):")
print(f"  claude mcp add --transport http \\")
print(f"    --header \"Authorization: Bearer {api_key_masked}\" \\")
print(f"    mcp-gateway \"{MAAS_GW}/mcp/mcp\"")

=== MaaS Gateway Mode ===

Cursor (.cursor/mcp.json):
{
  "mcpServers": {
    "mcp-gateway": {
      "url": "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/mcp",
      "headers": {
        "Authorization": "Bearer sk-oai-1Bq6X...bk0h"
      }
    }
  }
}

VS Code (.vscode/mcp.json):
{
  "servers": {
    "mcp-gateway": {
      "type": "http",
      "url": "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/mcp",
      "headers": {
        "Authorization": "Bearer sk-oai-1Bq6X...bk0h"
      }
    }
  }
}

Claude Code (CLI):
  claude mcp add --transport http \
    --header "Authorization: Bearer sk-oai-1Bq6X...bk0h" \
    mcp-gateway "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/mcp"


---
## 5. Verify Model Connectivity

In [2]:
import urllib.request, ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print("=== Model Endpoint Check ===")
print("")
try:
    model_url = f"{MODEL_ENDPOINT}/v1/models"
    headers = {"Content-Type": "application/json"}
    if MAAS_API_KEY:
        headers["Authorization"] = f"Bearer {MAAS_API_KEY}"
    req = urllib.request.Request(model_url, headers=headers)
    with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
        data = json.loads(resp.read())
        models = [m["id"] for m in data.get("data", [])]
        print(f"  [PASS    ] Model endpoint reachable")
        print(f"             Available models: {models}")
except urllib.error.HTTPError as e:
    print(f"  [FAIL    ] HTTP {e.code} — check model deployment or API key")
except Exception as e:
    print(f"  [FAIL    ] {e}")

print("")
print("IDE model verification:")
print("  Cursor:      Settings → Models → select model → test chat")
print("  VS Code:     Chat model picker → select RHOAI model → test")
print("  Claude Code: Launch with env vars → test response")

=== Model Endpoint Check ===

  [PASS    ] Model endpoint reachable
             Available models: ['qwen36-27b']

IDE model verification:
  Cursor:      Settings → Models → select model → test chat
  VS Code:     Chat model picker → select RHOAI model → test
  Claude Code: Launch with env vars → test response


---
## Summary

| IDE | Model Config | Status |
|-----|-------------|--------|
| **Cursor** | Settings → Models → OpenAI API Key + Override Base URL | Working (Pro+ required) |
| **VS Code** | `.vscode/chatLanguageModels.json` (BYOK) | Working |
| **Claude Code** | Env vars (`ANTHROPIC_BASE_URL` + `ANTHROPIC_AUTH_TOKEN`) | ⚠️ Model: blocked by vLLM 0.18.x / MCP: working |

> **MCP server registration** → `../1_mcp_servers/4_connect_ide_clients.ipynb`

### Key Points

- Cursor and VS Code connect to the **same self-hosted model** — no external API keys needed
- Claude Code model integration requires vLLM >= 0.23.0 (RHOAI 3.4 ships 0.18.x) — MCP tools still work
- MaaS Gateway (Phase 2) provides unified auth + rate limiting for all endpoints

## Next Steps

- `2_run_public_coding_assistant.ipynb` — Run with all 5 MCP tools (internet required)
- `3_run_closed_coding_assistant.ipynb` — Run with 3 local tools only (air-gapped)
- `../2_maas/2_enable_maas.ipynb` — Enable MaaS for production API key management